# Tutorial 7b: Bù đắp Dữ liệu (Data Imputation)

Nội dung này bao gồm:

* **Phương pháp Xóa bỏ (The deletion approach)**
    * Xóa bỏ các đặc trưng (cột) không đầy đủ
    * Xóa bỏ các thực thể (hàng) không đầy đủ

* **Sử dụng Pandas**
    * Bù đắp đơn giản bằng Pandas
    * Bù đắp bằng phương pháp nội suy (Interpolation) dùng Pandas

* **Sử dụng Sklearn**
    * Bù đắp đơn giản bằng Sklearn
    * Bù đắp dựa trên **KNN** (K-Nearest Neighbors) dùng Sklearn
    * Bù đắp lặp (Iterative imputation) dùng Sklearn

* **Áp dụng các mô hình đã học cho dữ liệu kiểm thử (test data) không đầy đủ**

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## Loading and exploring the data

In [3]:
import pandas as pd
# Hoặc tải dữ liệu Titanic đã được chia thành tập dữ liệu huấn luyện (train) và kiểm tra (test) theo https://www.kaggle.com/c/titanic/data
# Nhưng dữ liệu kiểm tra của Kaggle không có nhãn (labels)
# Do đó, chúng ta sẽ tải toàn bộ dữ liệu từ một kho lưu trữ dữ liệu rồi chia tách sau
titanic_data = pd.read_csv("https://www.openml.org/data/get_csv/16826755/phpMYEkMl.csv", na_values=['?']) #yo
titanic_data.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


**Các Giá trị được coi là "bị thiếu" (Values considered “missing”)**

Có nhiều cách để biểu diễn các giá trị bị thiếu, cả trong tệp dữ liệu lẫn trong thư viện **pandas** của Python.

Các giá trị bị thiếu trong dữ liệu có thể là các mục trống (blank entries), hoặc dấu **'?'**, hoặc một ký hiệu nào đó mà những người thu thập dữ liệu đã thống nhất để biểu thị dữ liệu không quan sát được (unobserved data).
Trong trường hợp này, đó là dấu **'?'** – khi biết điều này, chúng ta sẽ cho **`pandas`** biết giá trị nào được coi là bị thiếu thông qua đối số **`na_values=['?']`**.

Ở "đầu kia", **`pandas`** có thể biểu diễn các giá trị bị thiếu theo nhiều cách khác nhau. Như đã thấy ở trên, **"NaN"** là ký hiệu đánh dấu giá trị bị thiếu mặc định. Tuy nhiên, chúng ta cần có khả năng dễ dàng phát hiện giá trị này với dữ liệu thuộc các kiểu khác nhau: số dấu phẩy động (floating point), số nguyên (integer), boolean và đối tượng chung (general object). Tuy nhiên, trong nhiều trường hợp, một số dạng khác cũng có thể đề cập đến các giá trị bị thiếu như **`None`**, **“missing”** (bị thiếu), **“not available”** (không có sẵn), **“NA"**, hoặc **`(-)inf`** (vô cực âm hoặc dương).

In [4]:
# Hãy loại bỏ (drop) một số đặc trưng (features) mà chúng ta sẽ không xem xét ở đây.
titanic_data.drop(['name','ticket', 'embarked', 'boat' ,'body' ,'home.dest'], axis=1, inplace=True)

Bây giờ chúng ta sẽ chia dữ liệu thành các tập con **huấn luyện (train)** và **kiểm thử (test)** vì **CHỈ** dữ liệu huấn luyện mới được sử dụng để học các bộ bù đắp (imputers), sau đó các mô hình đã học được sẽ được áp dụng cho dữ liệu kiểm thử.

In [6]:
from sklearn.model_selection import train_test_split
y=titanic_data['survived']
X=titanic_data.drop(['survived'], axis=1)
X_titanic_train, X_titanic_test, y_titanic_train, y_titanic_test = train_test_split(X, y, test_size=0.3, random_state=42)

from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier()
#classifier=SVC()
classifier.fit(X_titanic_train, y_titanic_train)

# Có một vấn đề là một số đặc trưng chứa các giá trị chuỗi (string values), cụ thể là các đặc trưng "gender" (giới tính), "customer type" (loại khách hàng), "type of travel" (loại hình du lịch), "class" (hạng) và "satisfaction" (mức độ hài lòng). Do đó, hãy mã hóa (encode) các đặc trưng này.

In [8]:
import sklearn
!pip install -U scikit-learn
print('The scikit-learn version is {}.'.format(sklearn.__version__))

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   -------------------------------

ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\ASUS\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Users\ASUS\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\ASUS\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\ASUS\anaconda3\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ASUS\anaconda3\Lib\http\client.py", line 479, in read
    s = self.fp.read(amt)
        ^^^^^^^^^^^^^^^^^
  File "C:\Users\ASUS\anaconda3\Lib\socket.py", line 708, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^

In [9]:
import numpy as np
# Mã hóa các đặc trưng phân loại (categorical features) đồng thời bảo toàn các giá trị thiếu (missing values) trong các đặc trưng không hoàn chỉnh
from sklearn.preprocessing import OrdinalEncoder

# Khởi tạo bộ mã hóa cho đặc trưng 'sex'
encoder_sex = OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value=np.nan)

# Tạo một bản sao của dữ liệu huấn luyện để mã hóa
X_titanic_train_encoded=X_titanic_train.copy()

# Mã hóa đặc trưng 'sex'
X_titanic_train_encoded['sex'] = encoder_sex.fit_transform(X_titanic_train_encoded['sex'].values.reshape(-1, 1))

# Bây giờ, hãy mã hóa đặc trưng 'cabin' không hoàn chỉnh
# Khởi tạo bộ mã hóa cho đặc trưng 'cabin'
# (Bạn có thể sử dụng cùng một bộ mã hóa cho cả hai nhưng chúng tôi dùng hai bộ khác nhau để dễ hiểu hơn)
encoder_cabin = OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value=np.nan)

# Mã hóa đặc trưng 'cabin'. Chuyển đổi sang chuỗi (str) để OrdinalEncoder coi 'nan' là một danh mục hợp lệ.
X_titanic_train_encoded['cabin'] = encoder_cabin.fit_transform(X_titanic_train_encoded['cabin'].values.reshape(-1, 1).astype(str))

# Lấy mã số được gán cho giá trị "nan" (vì nó được mã hóa như một danh mục hợp lệ) của đặc trưng 'cabin'
cabin_nan_code=encoder_cabin.transform([['nan']])[0][0]
#print(cabin_nan_code)

# Bây giờ, khôi phục lại các giá trị 'nan' để chúng trở thành thiếu (missing) trong dữ liệu đã mã hóa
X_titanic_train_encoded['cabin'].replace(cabin_nan_code,np.nan,inplace=True)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_15200\747064947.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_titanic_train_encoded['cabin'].replace(cabin_nan_code,np.nan,inplace=True)


## `X_titanic_train_encoded` là Dữ liệu Huấn luyện Đã mã hóa và Không đầy đủ

**Lưu ý:** Tên biến này cho thấy đây là dữ liệu huấn luyện (train data) đã được **mã hóa (encoded)** để xử lý các đặc trưng chuỗi và **không đầy đủ (incomplete)**, tức là vẫn còn các giá trị bị thiếu (missing values).

In [10]:
# Kiểm tra các kiểu dữ liệu của dữ liệu đã được mã hóa, đảm bảo không còn đặc trưng nào thuộc kiểu object
X_titanic_train_encoded.info()

<class 'pandas.core.frame.DataFrame'>
Index: 916 entries, 1214 to 1126
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   pclass  916 non-null    int64  
 1   sex     916 non-null    float64
 2   age     729 non-null    float64
 3   sibsp   916 non-null    int64  
 4   parch   916 non-null    int64  
 5   fare    915 non-null    float64
 6   cabin   204 non-null    float64
dtypes: float64(4), int64(3)
memory usage: 57.2 KB


In [11]:
# Vì dữ liệu không còn giá trị chuỗi/object, hãy thử thực hiện phân loại (classification) bằng cách sử dụng dữ liệu đã mã hóa
classifier.fit(X_titanic_train_encoded, y_titanic_train)

RandomForestClassifier()

## Lưu ý Lỗi: ValueError: Input contains NaN, infinity or a value too large for dtype('float32').

Chúng ta cần xử lý các giá trị bị thiếu (**missing values** - NaN), giá trị vô cực (**infinity**) hoặc một giá trị quá lớn cho kiểu dữ liệu (**dtype('float32')**) trước khi thực hiện phân loại.

In [12]:
print("The number of missing values ")
print(X_titanic_train_encoded.isnull().sum())

The number of missing values 
pclass      0
sex         0
age       187
sibsp       0
parch       0
fare        1
cabin     712
dtype: int64
